<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# DATA/MSML 641: Natural Language Processing
## Lecture 8: Retrieval Augmented Generation

**University of Maryland, College Park**  
**Spring 2026**  
**Instructor**: Armin Mehrabian  
**Date**: March 17–18, 2026  


## Learning Objectives

By the end of this lecture, you will be able to:
1. Understand question answering systems in NLP
2. Explain the role of LLMs as knowledge bases and zero-shot learners
3. Describe the RAG (Retrieval Augmented Generation) architecture
4. Implement information retrieval methods (sparse and dense)
5. Evaluate RAG systems using appropriate metrics
6. Address challenges in RAG implementation including hallucination and privacy

# Question Answering in NLP

## What is Question Answering?

- Systems that answer questions posed in natural language
- **Applications**: Search Engines, AI Assistants, Chatbots...
- Can question anything: extremely high upper bounds on difficulty

## Types of Questions and Contexts

### Factoid Questions vs Non-Factoid Questions

**Factoid Questions:**
- Answers:
  - A short span of text/paragraph
  - Yes/No
  - A database entry
  - A list

**Non-Factoid Questions:**
- Context:
  - A passage, a document, a large collection of documents
  - Knowledge base
  - Semi-structured tables
  - Images
  - The web

### Example: Factoid Question Answering

**Passage Sentence:**
> In meteorology, precipitation is any product of the condensation of atmospheric water vapor that falls under gravity.

**Question:**
> What causes precipitation to fall?

**Answer Candidate:**
> gravity

**Word Relationships:**
- Between question and answer:
  - cause---gravity
  - precipitation---gravity
  - fall---gravity
  - what---gravity

## 2011: IBM Watson Beat Jeopardy Champions

**System Components:**
1. Question processing
2. Candidate answer generation
3. Candidate answer scoring
4. Confidence merging and ranking

This was a landmark achievement demonstrating that AI systems could compete with humans on complex question answering tasks.

## Textual Question Answering (Reading Comprehension)

### Stanford Question Answering Dataset (SQuAD)

- Human annotated questions derived from popular Wikipedia passages
- Recent models beat human baseline

**Example:**

*Passage:* "In meteorology, precipitation is any product of the condensation of atmospheric water vapor that falls under gravity. The main forms of precipitation include drizzle, rain, sleet, snow, graupel and hail... Precipitation forms as smaller droplets coalesce via collision with other rain drops or ice crystals within a cloud. Short, intense periods of rain in scattered locations are called 'showers'."

*Questions:*
- What causes precipitation to fall? → **gravity**
- What is another main form of precipitation besides drizzle, rain, snow, sleet and hail? → **graupel**
- Where do water droplets collide with ice crystals to form precipitation? → **within a cloud**

## 🔧 Simple Question Answering Demo

Let's implement a basic extractive QA system using pattern matching.

In [ ]:
# Simple Question Answering Demo
import re

print("="*60)
print("Simple Extractive Question Answering")
print("="*60)

# Sample passage
passage = """
In meteorology, precipitation is any product of the condensation of atmospheric
water vapor that falls under gravity. The main forms of precipitation include
drizzle, rain, sleet, snow, graupel and hail. Precipitation forms as smaller
droplets coalesce via collision with other rain drops or ice crystals within a cloud.
Short, intense periods of rain in scattered locations are called showers.
"""

print("\nPassage:")
print("-" * 60)
print(passage.strip())

# Questions and expected answers
questions = [
    ("What causes precipitation to fall?", "gravity"),
    ("What are the main forms of precipitation?", "drizzle, rain, sleet, snow, graupel and hail"),
    ("Where do droplets coalesce?", "within a cloud"),
]

print("\n" + "="*60)
print("Question Answering")
print("="*60)

def simple_qa(passage, question):
    """Simple pattern-based extractive QA"""
    passage_lower = passage.lower()
    question_lower = question.lower()

    # Simple heuristics
    if "what causes" in question_lower and "to fall" in question_lower:
        match = re.search(r"falls under (\w+)", passage_lower)
        if match:
            return match.group(1)

    if "main forms" in question_lower:
        match = re.search(r"include ([^.]+)\.", passage_lower)
        if match:
            return match.group(1).strip()

    if "where do" in question_lower and "coalesce" in question_lower:
        match = re.search(r"coalesce[^.]*?within ([^.]+)\.", passage_lower)
        if match:
            return match.group(1).strip()

    return "Answer not found"

for question, expected in questions:
    answer = simple_qa(passage, question)
    print(f"\nQ: {question}")
    print(f"A: {answer}")
    print(f"Expected: {expected}")
    print(f"Match: {'✓' if answer.lower() in expected.lower() else '✗'}")

print("\n✅ This is a toy example!")
print("   Modern QA systems use:")
print("   - BERT-based models for span extraction")
print("   - Dense retrieval for finding relevant passages")
print("   - LLMs for generation-based QA (like RAG)")


## Other Types of Question Answering

- **Conversational Question Answering (CoQA)**
- **Multi-task, specialized domain knowledge (MMLU)**
- **Multi-hop reasoning questions (HotpotQA)**
- **Long-form Question Answering (ELI5: Long Form Question Answering)**
- **Open-domain Question Answering** (Reading Wikipedia to Answer Open-Domain Questions)
- **Knowledge Base Question Answering** (Semantic Parsing on Freebase from Question-Answer Pairs)
- **Visual Question Answering (VQA)**

## Why Do We Want to Do Question Answering?

- Reading comprehension is an important testbed for evaluating how well computer systems understand human language

- **Wendy Lehnert, 1977:** "Since questions can be devised to query any aspect of text comprehension, the ability to answer questions is the strongest possible demonstration of understanding."

- Many other NLP tasks can be reduced to a reading comprehension problem:
  - Information extraction
  - Summarization
  - Semantic role labeling

## Feature-Based Methods for QA

Used in SQuAD paper (Rajpurkar et al, 2016):

1. Generate a list of candidate answers $(a_1, a_2, ..., a_n)$
2. Define a feature vector $\phi(p, q, a) \in \mathbb{R}^d$:
   - Word/bigram features
   - Parse trees
   - Dependency labels, length, part-of-speech tags
3. Apply a multi-class logistic regression model to predict the true answer

## BERT for Reading Comprehension

BERT is pre-trained on two training objectives:
- **Masked language model (MLM)**
- **Next sentence prediction (NSP)**

BERTlarge has 24 layers and 330M parameters

### Training Objective

$$\mathcal{L} = -\log p_{\text{start}}(s^*) - \log p_{\text{end}}(e^*)$$

$$p_{\text{start}}(i) = \text{softmax}_i(\mathbf{w}_{\text{start}}^\top \mathbf{h}_i)$$

$$p_{\text{end}}(i) = \text{softmax}_i(\mathbf{w}_{\text{end}}^\top \mathbf{h}_i)$$

where $\mathbf{h}_i$ is the hidden vector of $c_i$, returned by BERT

- $s^*$ is the true start position
- $e^*$ is the true end position
- $p_{\text{start}}$ and $p_{\text{end}}$ are probability distributions over all possible positions

# Natural Language Generation

## Language Models as Next Word Predictors

- **Language Models (LMs)** give probability over sentences...

$$p(x_1, \ldots, x_n) = \prod_{i=1}^n p(x_i | x_{<i})$$

- ...by giving probabilities of individual words

- **LLMs as 'next word predictors'**

- **Question Answering**: Is the parametric memory of LLMs enough?

## Using LLMs for Question Answering

- We can all use LLMs to answer questions

$$p(x_1, \ldots, x_n) = \prod_{i=1}^n p([Q:]; q; [A:]; x_{<i})$$

- LLM will tell us what is 'most likely' answer
- Standard "Prompting" approach
- **Question: Why does this work?**

**Example:**
> *Query:* "Who is the most recent winner of the World Series?"
>
> *Response:* "In 2023, the Texas Rangers won their first World Series championship in franchise history, defeating the Arizona Diamondbacks in 5 games. Since my knowledge cutoff is April 2024 and the 2024 World Series hasn't happened yet from that perspective, this is the most recent World Series winner I can definitively confirm."

## LLMs are... Zero-Shot Learners

- LMs were traditionally used in a supervised learning setting for NLP tasks
- As parameters increased, they may be used in "zero-shot" settings (Kojima 2023)

**Performance scales with model size:**

| Model Size | Zero-Shot | One-Shot | Few-Shot (K=64) |
|------------|-----------|----------|------------------|
| 0.1B       | ~7%       | ~7%      | ~7%              |
| 0.4B       | ~7%       | ~13%     | ~15%             |
| 0.8B       | ~15%      | ~14%     | ~28%             |
| 1.3B       | ~20%      | ~21%     | ~43%             |
| 2.6B       | ~32%      | ~36%     | ~52%             |
| 6.7B       | ~40%      | ~46%     | ~58%             |
| 13B        | ~44%      | ~53%     | ~70%             |
| 175B       | ~70%      | ~71%     | ~72%             |

## LLMs are... Knowledge Bases

- LLMs learn co-occurrences from text
- Correct facts will appear often enough for LLMs to (sometimes) learn them
- Scales with training data and parameters
- They will be biased (Rogers 2020)

### Knowledge Access Comparison

**Symbolic KB:**
- Query: (DANTE, born-in, X)
- Symbolic KB Memory Access → FLORENCE

**Neural LM:**
- "Dante was born in [MASK]."
- Neural LM Memory Access (e.g. ELMo/BERT) → Florence

## LLMs are... Scaling

- More parameters, better performance of LLMs
- Larger models have more:
  - training data
  - storage space
  - reasoning ability

**Performance improves with scale:**
- Models with 10^11 parameters (PaLM 5-shot) approach human performance on many tasks
- GPT-3 series shows consistent improvement with parameter count

## LLMs are....

### Strengths:
- **Zero-Shot learners** (Huang 2023)
  - Reasoning tools
- **Knowledge Bases**
  - Parametric knowledge bases
- **Scaling**

### But also:
- **Biased**
- **Hallucinators**
- **Frozen** (knowledge cutoff)
- **Not private**

$$p(x_1, \ldots, x_n) = \prod_{i=1}^n p([Q:]; q; [A:]; x_{<i})$$

**Example of knowledge cutoff:**
> *Query:* "Who is the most recent winner of the World Series?"
>
> *Issue:* Model knowledge is frozen at training time, so recent events are unknown.

# Addressing LLM Limitations with External Memory

## How Can We Get More Timely or Relevant Answers with LLMs?

- **Condition generation on external information:**
  - Knowledge Bases
  - Web-search
  - Wikipedia
  - **"Non-parametric memory"**

- Conditioning directly on KB is complex

- Easier to "cast" a small, relevant part of the Non-parametric memory to text

- Add this additional context "c" to input

- **Conditional generation on ground-truth knowledge** $P(X|c)$

## Using External Memory: Example

- Insert retrieval between prompt and LLM Generation
- Retrieval returns additional non-parametric context
- Concatenate this to the prompt then run inference
- Retrieval is likely to be relevant
- **Question: how could it fail?**

### Example: World Series Winners

**Retrieved Context from Wikipedia:**
```
2019 Washington Nationals (1, 1-0)
2020 Los Angeles Dodgers (21, 7-14)
2021 Atlanta Braves (10, 4-6)
2022 Houston Astros (5, 2-3)
2023 Texas Rangers (3, 1-2)
2024 Los Angeles Dodgers (22, 8-14)
```

**Enhanced Response:**
> "The Los Angeles Dodgers won the 2024 World Series. This was their 22nd World Series appearance and 8th championship victory."

## Limitations of External Memory

- Insert retrieval between prompt and LLM Generation
- Retrieval returns additional non-parametric context
- Concatenate this to the prompt then run inference
- Retrieval is likely to be relevant
- **Do we know that LLMs will use the contextual knowledge?**
- **NO:** Only if the LLM determines that the "context is relevant" (Farahani 2024)

### Example: Model Ignoring Retrieved Context

**Retrieved Context:**
```
2023 Texas Rangers (3, 1-2)
```

**Problematic Response:**
> "The Texas Rangers won their first World Series championship in 2023, defeating the Arizona Diamondbacks. The Rangers were established in 1961 and reached their first World Series victory after 63 years."
>
> "Since you shared information showing a 2024 Dodgers victory - I should note that my knowledge cutoff is April 2024, so I can't verify World Series results after that date."

*Model may doubt or ignore retrieved information if it conflicts with internal knowledge!*

# The RAG Recipe

## What is Retrieval Augmented Generation?

Want to "proceduralise" this process of **search + generate**

### **Retrieval Augmented Generation (RAG)**

**Steps:**
1. Build document database (open, commercial, etc.)
2. Accept a query
3. **Indexing**: Process documents
4. **Retrieval**: Surfaces relevant documents from database
5. **Prompt**: Combine query with retrieved documents
6. **Generator** (Frozen LLM): Responds to query with documents
7. **Output**: Final answer

**Note:** Query is used twice - for retrieval and for generation

## RAG Architecture

**Retrieval Augmented Generation: RAG**

- Retrieval + Generation →
- **Information Retrieval** + **Natural Language Generation**
- Advantages of both
- Complexities of both
- Also new complexities!

### Visual Overview

```
Q: When was the premiere of The Magic Flute?
         ↓
    [Retriever]
    Indexed Docs
         ↓
  Relevant Docs → [Reader/Generator] → A: 1791
                  (LLM with prompt)
```

## Components of a RAG System

1. **Document Collection**: Corpus of knowledge
2. **Retriever**: Finds relevant documents/passages
3. **Prompt Engineering**: Structures input for LLM
4. **Language Model**: Generates answers from context
5. **Output Processing**: Formats and refines responses
6. **Evaluation Metrics**: Measures system performance

## Simple RAG Implementation Example

Using LlamaIndex:

In [ ]:
# Example using LlamaIndex
# Uncomment to run (requires llama_index installation)

# from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# # Load documents
# documents = SimpleDirectoryReader("data").load_data()

# # Create index
# index = VectorStoreIndex.from_documents(documents)

# # Create query engine
# query_engine = index.as_query_engine()

# # Query
# response = query_engine.query("Who first proposed the perceptron algorithm?")
# print(response)

# Information Retrieval in RAG

## R in RAG - Information Retrieval

- Field in its own right, at least 80 years old (Bush, 1945, Mooers 1950)
- **Task:** finding resources relevant to an information need

### Units:

- **Query**: User's expression of information need
- **Document**: Unit of text being searched
- **Collection**: Set of searchable documents
- **Term**: Word or phrase in collection
- **Ranking**: Ordering documents by relevance
- **Retrieval**: Finding relevant documents for query

**Question:** Given a query, how can we search for relevant documents?

## Retrieval Methods

1. **Sparse retrieval** (TF-IDF, BM25)
2. **Document-level retrieval** (Dense vectors)
3. **Token-level** retrieval
4. **Cross-encoder** re-ranking
5. **Tool retriever** aka "just Google it"

## Sparse Retrieval: Terms

- **Intuition**: texts with similar terms are more similar
- Express a document as a vector of terms
- Unit-normalize
- Take inner product and sort

### Example: Shakespeare Plays

Plotting plays in 2D space based on word "battle" vs "fool":
- Henry V: high "battle", low "fool"
- Julius Caesar: medium "battle", low "fool"
- As You Like It: low "battle", high "fool"
- Twelfth Night: low "battle", very high "fool"

## Sparse Retrieval: Weighted Terms (TF-IDF)

### Term Frequency (TF)
- Frequency of term in document

$$\text{TF}(t,d) = \frac{\text{freq}(t,d)}{\sum_{t'} \text{freq}(t',d)}$$

### Inverse Document Frequency (IDF)
- Rarity of term across documents

$$\text{IDF}(t) = \log\left(\frac{|D|}{\sum_{d' \in D} \delta(\text{freq}(t,d') > 0)}\right)$$

### TF-IDF

$$\text{TF-IDF}(t,d) = \text{TF}(t,d) \times \text{IDF}(t)$$

- **Balances** term importance in doc/collection
- **Higher** for rare, frequent terms
- **Lower** for common or infrequent terms

### Example from Shakespeare

| Word     | df | idf   |
|----------|-------|-------|
| Romeo    | 1     | 1.57  |
| salad    | 2     | 1.27  |
| Falstaff | 4     | 0.967 |
| forest   | 12    | 0.489 |
| battle   | 21    | 0.246 |
| wit      | 34    | 0.037 |
| fool     | 36    | 0.012 |
| good     | 37    | 0     |
| sweet    | 37    | 0     |

## 💡 TF-IDF Intuition

**Goal**: Weight terms by importance
- **High TF-IDF**: Term appears often in THIS document but rare across corpus
- **Low TF-IDF**: Either rare in this doc OR common everywhere (like "the")

Let's calculate TF-IDF step-by-step for a toy corpus.

In [ ]:
# TF-IDF Worked Example
import numpy as np
import pandas as pd
from collections import Counter
import math

# Toy corpus: 4 documents
corpus = [
    "the dog chased the cat",           # Doc 0
    "the cat climbed the tree",         # Doc 1
    "the dog barked loudly",            # Doc 2
    "a bird sang in the tree"           # Doc 3
]

print("📚 Corpus:")
for i, doc in enumerate(corpus):
    print(f"  Doc {i}: {doc}")

# Step 1: Calculate Term Frequency (TF)
print("\n" + "="*60)
print("STEP 1: Term Frequency (TF)")
print("="*60)

def compute_tf(doc):
    """Compute term frequency: count / total_words"""
    words = doc.lower().split()
    word_count = Counter(words)
    total_words = len(words)
    tf = {word: count / total_words for word, count in word_count.items()}
    return tf

tf_docs = [compute_tf(doc) for doc in corpus]

print("\nTF for Doc 0:", corpus[0])
for word, freq in sorted(tf_docs[0].items()):
    print(f"  {word:10} → {freq:.3f}")

# Step 2: Calculate Document Frequency (DF)
print("\n" + "="*60)
print("STEP 2: Document Frequency (DF)")
print("="*60)

all_words = set()
for doc in corpus:
    all_words.update(doc.lower().split())

df = {}
for word in all_words:
    df[word] = sum(1 for doc in corpus if word in doc.lower())

print("\nDocument Frequency:")
for word, freq in sorted(df.items(), key=lambda x: -x[1]):
    print(f"  {word:10} → appears in {freq}/{len(corpus)} documents")

# Step 3: Calculate Inverse Document Frequency (IDF)
print("\n" + "="*60)
print("STEP 3: Inverse Document Frequency (IDF)")
print("="*60)

N = len(corpus)
idf = {word: math.log(N / df_val) for word, df_val in df.items()}

print("\nIDF (higher = more discriminative):")
for word, idf_val in sorted(idf.items(), key=lambda x: -x[1]):
    print(f"  {word:10} → log({N}/{df[word]}) = {idf_val:.3f}")

# Step 4: Calculate TF-IDF
print("\n" + "="*60)
print("STEP 4: TF-IDF = TF × IDF")
print("="*60)

def compute_tfidf(tf, idf):
    return {word: tf_val * idf[word] for word, tf_val in tf.items()}

tfidf_docs = [compute_tfidf(tf, idf) for tf in tf_docs]

print("\nTF-IDF for Doc 0:", corpus[0])
for word, score in sorted(tfidf_docs[0].items(), key=lambda x: -x[1]):
    print(f"  {word:10} → {tf_docs[0][word]:.3f} × {idf[word]:.3f} = {score:.4f}")

# Step 5: Create TF-IDF Matrix
print("\n" + "="*60)
print("STEP 5: TF-IDF Matrix (rows=docs, cols=words)")
print("="*60)

# Create matrix
vocab = sorted(all_words)
tfidf_matrix = np.zeros((len(corpus), len(vocab)))

for doc_idx, tfidf in enumerate(tfidf_docs):
    for word_idx, word in enumerate(vocab):
        if word in tfidf:
            tfidf_matrix[doc_idx, word_idx] = tfidf[word]

df_tfidf = pd.DataFrame(tfidf_matrix, columns=vocab,
                        index=[f"Doc{i}" for i in range(len(corpus))])
print("\n", df_tfidf.round(3))

print("\n✅ Key Insight:")
print("   - 'the' has TF-IDF ≈ 0 (common everywhere → IDF ≈ 0)")
print("   - 'dog', 'cat' have higher TF-IDF (more discriminative)")


## BM25: A Modern Weighting Scheme

- Extension of TF-IDF
- Two tunable parameters: **k** and **b**
- **k** adjusts impact of term frequency
- **b** controls document length normalization
- **Outperforms basic TF-IDF scoring**
- **Can outperform dense (neural) methods**
- **Vocabulary mismatch problem**

### Formula

$$\sum_{t \in q} \log\left(\frac{N}{df_t}\right) \frac{tf_{t,d}}{k\left(1-b+b\left(\frac{|d|}{|d_{\text{avg}}|}\right)\right) + tf_{t,d}}$$

where:
- N = total documents
- $df_t$ = document frequency of term t
- $tf_{t,d}$ = term frequency in document
- $|d|$ = document length
- $|d_{\text{avg}}|$ = average document length

## 💡 BM25 Intuition

**BM25 improves TF-IDF with**:
- **Saturation**: Multiple occurrences don't keep increasing score linearly
- **Length normalization**: Penalizes long documents (parameter `b`)
- **Tunable parameters**: `k` controls saturation, `b` controls length effect

**Formula**:
$$\sum_{t \in q} \log\left(\frac{N}{df_t}\right) \frac{tf_{t,d}}{k\left(1-b+b\left(\frac{|d|}{|d_{\text{avg}}|}\right)\right) + tf_{t,d}}$$

Let's see how BM25 works with actual numbers!

In [ ]:
# BM25 Worked Example
import numpy as np
import pandas as pd

# Same corpus as TF-IDF example
corpus = [
    "the dog chased the cat",
    "the cat climbed the tree",
    "the dog barked loudly",
    "a bird sang in the tree"
]

print("📚 Corpus:")
for i, doc in enumerate(corpus):
    print(f"  Doc {i}: {doc}")

# BM25 parameters (common defaults)
k = 1.5  # Controls term frequency saturation
b = 0.75  # Controls length normalization

print(f"\nBM25 Parameters: k={k}, b={b}")

# Compute document stats
doc_lengths = [len(doc.split()) for doc in corpus]
avgdl = np.mean(doc_lengths)

print(f"\nDocument lengths: {doc_lengths}")
print(f"Average document length: {avgdl:.2f}")

# Tokenize
docs_tokens = [doc.lower().split() for doc in corpus]

# Build document frequency
from collections import Counter
all_words = set()
for tokens in docs_tokens:
    all_words.update(tokens)

df_bm25 = {}
for word in all_words:
    df_bm25[word] = sum(1 for tokens in docs_tokens if word in tokens)

# Query: "dog cat"
query = "dog cat"
query_terms = query.lower().split()

print(f"\n" + "="*60)
print(f"Query: '{query}'")
print("="*60)

N = len(corpus)

# Calculate BM25 score for each document
def bm25_score(doc_tokens, query_terms, doc_length, avgdl, df, N, k, b):
    score = 0
    tf = Counter(doc_tokens)

    for term in query_terms:
        if term not in df:
            continue

        # IDF component
        idf = np.log((N - df[term] + 0.5) / (df[term] + 0.5) + 1)

        # TF component with saturation and length norm
        tf_val = tf[term]
        numerator = tf_val * (k + 1)
        denominator = tf_val + k * (1 - b + b * (doc_length / avgdl))
        tf_component = numerator / denominator

        term_score = idf * tf_component
        score += term_score

    return score

scores = []
for i, tokens in enumerate(docs_tokens):
    score = bm25_score(tokens, query_terms, doc_lengths[i], avgdl, df_bm25, N, k, b)
    scores.append(score)
    print(f"\nDoc {i}: {corpus[i]}")
    print(f"  Length: {doc_lengths[i]}, BM25 Score: {score:.4f}")

# Rank documents
ranked = sorted(enumerate(scores), key=lambda x: -x[1])

print(f"\n" + "="*60)
print("RANKED RESULTS")
print("="*60)
for rank, (doc_idx, score) in enumerate(ranked, 1):
    print(f"  {rank}. Doc {doc_idx} (score={score:.4f}): {corpus[doc_idx]}")

print("\n✅ Key Insights:")
print("   - Doc 0 ranks highest (contains both 'dog' and 'cat')")
print("   - Doc 2 ranks second (contains 'dog')")
print("   - Length normalization prevents long docs from dominating")
print("   - Saturation: 2nd occurrence of 'the' doesn't double the score")


## Dense Vector Representations for IR

- **Addresses vocabulary mismatch problem**
- Enables similarity beyond exact word matches
- Uses neural embeddings instead of word counts
- Encodes semantic meaning of text
- BERT and other encoder-based Transformers commonly used

**Question:** How can we generate vector representations of entire sentences?

### Two Main Architectures:
1. **Cross-encoder**: Joint encoding
2. **Bi-encoder**: Separate encoding

## Bi-Encoder Architecture

- Separate encoders for query and document
- **Query**: $z_q = \text{BERT}(q)$
- **Document**: $z_d = \text{BERT}(d)$
- **Similarity**: $\text{score}(q, d) = z_q \cdot z_d$

### Advantages:
- **Efficient**: Documents pre-encoded once, ahead-of-time
- Fast similarity computation (cosine similarity)

### Disadvantages:
- **Approximate**: Less accurate than full interaction models
- No direct interaction between query and document tokens

### Architecture Diagram:
```
    cosine-sim(q, d)
         ↑    ↑
         q    d
         ↑    ↑
     pooling  pooling
         ↑    ↑
       BERT  BERT
         ↑    ↑
       Query  Doc
```

Output: Similarity score in range [-1, 1]

## 🔧 Bi-Encoder Implementation

Let's implement a simple bi-encoder using sentence transformers.

In [ ]:
# Bi-Encoder Demonstration
# Note: Install with: pip install sentence-transformers

try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity

    print("="*60)
    print("Bi-Encoder Demonstration with Sentence Transformers")
    print("="*60)

    # Load pre-trained model
    model = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast model

    # Example documents
    documents = [
        "The dog chased the cat in the garden",
        "A puppy ran after a kitten outdoors",
        "Python is a programming language",
        "Machine learning uses neural networks",
    ]

    # Query
    query = "pet animals playing outside"

    print(f"\nQuery: '{query}'")
    print(f"\nDocuments:")
    for i, doc in enumerate(documents):
        print(f"  {i}. {doc}")

    # Encode query and documents separately (bi-encoder approach)
    print("\n" + "="*60)
    print("Encoding with Bi-Encoder...")
    print("="*60)

    query_embedding = model.encode([query])
    doc_embeddings = model.encode(documents)

    print(f"\nQuery embedding shape: {query_embedding.shape}")
    print(f"Document embeddings shape: {doc_embeddings.shape}")
    print(f"Embedding dimension: {query_embedding.shape[1]}")

    # Calculate similarities
    print("\n" + "="*60)
    print("Computing Cosine Similarities")
    print("="*60)

    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

    # Rank documents
    ranked_indices = np.argsort(similarities)[::-1]

    print(f"\nRanked Results:")
    for rank, idx in enumerate(ranked_indices, start=1):
        print(f"  {rank}. Doc {idx} (similarity={similarities[idx]:.4f}): {documents[idx]}")

    print("\n✅ Observations:")
    print("   - Docs 0 and 1 rank highest (semantic match with query)")
    print("   - Docs 2 and 3 rank lower (different topic)")
    print("   - Bi-encoder captures semantic similarity beyond word overlap")
    print("   - Documents can be encoded once and reused for many queries!")

except ImportError:
    print("⚠️  sentence-transformers not installed")
    print("   Install with: pip install sentence-transformers")
    print("   This demo shows how bi-encoders work in production RAG systems")


## Cross-Encoder Architecture

- Shared cross-encoder for query and document
- Encodes entire text: [CLS] Query [SEP] Document [SEP]
- More nuanced than simple bi-encoder
- **Scales linearly with document count**
- **Used as a reranker** after reducing the pool of candidate matches with a Bi-Encoder

### Architecture Diagram:
```
     Classifier
         ↑
       BERT
      ↑    ↑
    Query  Doc
```

Output: Relevance score in range [0, 1]

### Advantages:
- Full interaction between query and document
- More accurate relevance scoring

### Disadvantages:
- Cannot pre-compute document embeddings
- Must encode query+doc pair for every document
- Much slower than bi-encoder

## Cosine Similarity for Document Ranking

- Measures similarity between document and query
- Calculates angle between two vectors
- Range from -1 (opposite) to 1 (identical)
- Normalizes for document length differences
- Higher cosine indicates greater relevance
- Efficient for high-dimensional sparse vectors

### Formula

$$\text{score}(q,d) = \cos(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{|\mathbf{q}||\mathbf{d}|}$$

**Question:** Are L2 and Cosine Distance measures monotonic? What if vectors are unit-normed?

**Answer:** For unit-normed vectors, L2 distance and cosine similarity are monotonically related!

## 💡 Cosine Similarity Intuition

**Idea**: Measure the angle between two vectors
- **1.0**: Identical direction (very similar)
- **0.0**: Orthogonal (unrelated)
- **-1.0**: Opposite direction

**Why normalize?** Long documents shouldn't automatically score higher!

Let's calculate step-by-step.

In [ ]:
# Cosine Similarity Worked Example
import numpy as np
import matplotlib.pyplot as plt

# Example: 2D vectors for visualization
print("="*60)
print("Example 1: 2D Vectors (for intuition)")
print("="*60)

# Two documents represented in 2D space (imagine words "cat" and "dog")
vec_a = np.array([3, 4])  # Doc A: 3 mentions of "cat", 4 of "dog"
vec_b = np.array([6, 8])  # Doc B: 6 mentions of "cat", 8 of "dog"
vec_c = np.array([4, 1])  # Doc C: 4 mentions of "cat", 1 of "dog"

print(f"\nVector A: {vec_a} (Doc A)")
print(f"Vector B: {vec_b} (Doc B)")
print(f"Vector C: {vec_c} (Doc C)")

# Calculate cosine similarity manually
def cosine_sim(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

# Step-by-step for A and B
print(f"\n" + "="*60)
print("Cosine Similarity: A vs B")
print("="*60)
dot_ab = np.dot(vec_a, vec_b)
norm_a = np.linalg.norm(vec_a)
norm_b = np.linalg.norm(vec_b)
cos_ab = dot_ab / (norm_a * norm_b)

print(f"Step 1: Dot product A·B = {vec_a}·{vec_b} = {dot_ab}")
print(f"Step 2: ||A|| = √(3² + 4²) = {norm_a:.3f}")
print(f"Step 3: ||B|| = √(6² + 8²) = {norm_b:.3f}")
print(f"Step 4: cos(A,B) = {dot_ab} / ({norm_a:.3f} × {norm_b:.3f}) = {cos_ab:.4f}")

print(f"\n✅ cos(A,B) = {cos_ab:.4f} → Nearly identical! (B is just 2×A)")

# A vs C
print(f"\n" + "="*60)
print("Cosine Similarity: A vs C")
print("="*60)
cos_ac = cosine_sim(vec_a, vec_c)
print(f"cos(A,C) = {cos_ac:.4f} → Somewhat similar (different proportions)")

# B vs C
print(f"\n" + "="*60)
print("Cosine Similarity: B vs C")
print("="*60)
cos_bc = cosine_sim(vec_b, vec_c)
print(f"cos(B,C) = {cos_bc:.4f} → Somewhat similar")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
origin = np.array([0, 0])

# Plot vectors
ax.quiver(*origin, *vec_a, angles='xy', scale_units='xy', scale=1, color='blue', width=0.006, label='Doc A [3,4]')
ax.quiver(*origin, *vec_b, angles='xy', scale_units='xy', scale=1, color='green', width=0.006, label='Doc B [6,8]')
ax.quiver(*origin, *vec_c, angles='xy', scale_units='xy', scale=1, color='red', width=0.006, label='Doc C [4,1]')

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_xlabel('"cat" mentions', fontsize=12)
ax.set_ylabel('"dog" mentions', fontsize=12)
ax.set_title('Cosine Similarity: Measuring Angle Between Vectors', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\n✅ Key Insight:")
print("   - A and B point in nearly the same direction → cos ≈ 1.0")
print("   - A and C point in different directions → cos < 1.0")
print("   - Cosine similarity measures DIRECTION, not magnitude!")


## Nearest Neighbour Lookup

- **Problem**: find closest vectors in high-dimensional space
- Exact search scales linearly O(nd)
- Approximate methods trade accuracy for speed
- Popular algorithms: LSH, HNSW, FAISS
- Essential for large-scale dense retrieval
- Enables efficient similarity search in RAG

### Distance Metrics:

- **Euclidean**: $d(x,y) = \sqrt{\sum_i (x_i - y_i)^2}$
- **Cosine**: $\text{sim}(x,y) = \frac{x \cdot y}{||x|| ||y||}$
- **Manhattan**: $d(x,y) = \sum_i |x_i - y_i|$

### Complexity:
- O(nd) search time, n vectors in d dimensions
- O(nd) storage space

## Approximate Nearest Neighbor Search

### Problem
- **Input**: Points $P = \{p_1, ..., p_n\}$, query $q$
- **Goal**: Find [approximate] nearest $p \in P$ to $q$
- **Challenge**: Want faster than O(n) lookup

### Methods

**LSH (Locality Sensitive Hashing)**
- Hash similar items to same bucket
- Query: $O(n^{1/(1+\epsilon)})$
- Space: $O(n^{1+1/(1+\epsilon)})$

**Trees (KD-tree, Ball-tree)**
- Partition space hierarchically
- Query: O(log n) average, O(n) worst
- Space: O(n)

**Random Projections**
- Project to random lower dimensions
- Query: O(d log n) expected
- Space: O(n)

# Evaluation Metrics for Information Retrieval

## Core Metrics

- **Precision**: Fraction of retrieved docs relevant
- **Recall**: Fraction of relevant docs retrieved
- **Mean Average Precision (MAP)**
- **Mean Reciprocal Rank (MRR)**
- **Normalized Discounted Cumulative Gain (NDCG)**

## Precision and Recall

### Precision
"When the model says yes, how often is it correct?"
- ↔ Type I Error (False Positives)

$$\text{Precision} = \frac{TP}{TP + FP}$$

### Recall
"Out of all correct items, how many did the model find?"
- ↔ Type II Error (False Negatives)

$$\text{Recall} = \frac{TP}{TP + FN}$$

where:
- TP = True Positives
- FP = False Positives
- FN = False Negatives

### Example: Retrieved Chunks
- **High precision**: All retrieved chunks have relevant content
- **High recall**: Most relevant content is retrieved

## 📊 Worked Example: Precision & Recall

**Scenario**: Retrieval system returns 10 documents for a query
- **Ground truth**: 6 relevant documents exist in the database
- **System retrieved**: 10 documents

Let's calculate!

In [ ]:
# Precision & Recall Worked Example
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Scenario: Retrieved 10 documents, 6 are truly relevant in database
print("="*60)
print("Scenario: Query returns 10 documents")
print("="*60)

# Ground truth relevance for the 10 retrieved docs
# 1 = relevant, 0 = not relevant
y_true = np.array([1, 1, 0, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])  # System retrieved all 10

print("\nRetrieved documents (in rank order):")
for i, (true, pred) in enumerate(zip(y_true, y_pred), 1):
    status = "✓ RELEVANT" if true == 1 else "✗ NOT RELEVANT"
    print(f"  Doc {i:2d}: {status} (retrieved by system)")

# Calculate metrics manually
TP = np.sum((y_true == 1) & (y_pred == 1))  # True Positives
FP = np.sum((y_true == 0) & (y_pred == 1))  # False Positives
FN = np.sum((y_true == 1) & (y_pred == 0))  # False Negatives
TN = np.sum((y_true == 0) & (y_pred == 0))  # True Negatives

print(f"\n" + "="*60)
print("STEP 1: Count outcomes")
print("="*60)
print(f"  True Positives (TP):  {TP} (relevant docs retrieved)")
print(f"  False Positives (FP): {FP} (irrelevant docs retrieved)")
print(f"  False Negatives (FN): {FN} (relevant docs NOT retrieved)")
print(f"  True Negatives (TN):  {TN} (irrelevant docs NOT retrieved)")

print(f"\n" + "="*60)
print("STEP 2: Calculate Precision")
print("="*60)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
print(f"  Precision = TP / (TP + FP)")
print(f"            = {TP} / ({TP} + {FP})")
print(f"            = {TP} / {TP + FP}")
print(f"            = {precision:.3f}")
print(f"\n  💡 Interpretation: {precision*100:.0f}% of retrieved docs are relevant")

print(f"\n" + "="*60)
print("STEP 3: Calculate Recall")
print("="*60)
# Note: Total relevant docs = TP + FN (what we retrieved + what we missed)
total_relevant = TP + FN
recall = TP / total_relevant if total_relevant > 0 else 0
print(f"  Recall = TP / (TP + FN)")
print(f"         = {TP} / ({TP} + {FN})")
print(f"         = {TP} / {total_relevant}")
print(f"         = {recall:.3f}")
print(f"\n  💡 Interpretation: We found {recall*100:.0f}% of all relevant docs")

print(f"\n" + "="*60)
print("STEP 4: F1 Score (Harmonic Mean)")
print("="*60)
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"  F1 = 2 × (Precision × Recall) / (Precision + Recall)")
print(f"     = 2 × ({precision:.3f} × {recall:.3f}) / ({precision:.3f} + {recall:.3f})")
print(f"     = {f1:.3f}")

# Visualize with confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Relevant', 'Relevant'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix\nPrecision={precision:.2f}, Recall={recall:.2f}, F1={f1:.2f}')
plt.tight_layout()
plt.show()

print("\n✅ Key Takeaways:")
print(f"   - High Recall ({recall:.1%}): We found most relevant docs")
print(f"   - Moderate Precision ({precision:.1%}): But retrieved some junk too")
print("   - Trade-off: Retrieving more → higher recall, lower precision")


## Mean Average Precision (MAP)

- Measures overall effectiveness of ranked retrieval results
- Captures both precision and position of relevant documents
- Calculated in multiple steps from top to bottom of results
- Averages precision scores at each relevant document position
- Higher MAP indicates better overall ranking quality
- Particularly useful for multi-document retrieval tasks

### Example
If relevant docs are at ranks 1, 4, 7 in 10 results:
- At rank 1: precision = 1/1 = 1.0
- At rank 4: precision = 2/4 = 0.5
- At rank 7: precision = 3/7 ≈ 0.43
- AP = (1.0 + 0.5 + 0.43)/3 ≈ 0.64

### Formula

$$\text{MAP} = \frac{1}{Q} \sum \text{AP}_q$$

where Q is number of queries and $\text{AP}_q$ is Average Precision for query q

## 📊 Mean Average Precision (MAP) - Detailed Calculation

MAP rewards systems that rank relevant documents higher.

**Formula**: Average the precision values at each relevant document position.

In [ ]:
# MAP (Mean Average Precision) Worked Example
import numpy as np

print("="*60)
print("Mean Average Precision (MAP) Example")
print("="*60)

# Scenario: 3 queries, each with ranked retrieval results
# Format: list of relevance labels (1=relevant, 0=not relevant) in rank order

queries = {
    "Query 1": [1, 1, 0, 1, 0, 0, 1, 0, 0, 0],  # 4 relevant docs at ranks 1,2,4,7
    "Query 2": [0, 1, 0, 0, 1, 0, 0, 0, 0, 0],  # 2 relevant docs at ranks 2,5
    "Query 3": [1, 0, 0, 0, 0, 1, 0, 0, 0, 1],  # 3 relevant docs at ranks 1,6,10
}

def calculate_average_precision(relevance_list):
    """Calculate Average Precision for a single query"""
    precisions = []
    num_relevant = 0

    for i, is_relevant in enumerate(relevance_list, start=1):
        if is_relevant == 1:
            num_relevant += 1
            precision_at_i = num_relevant / i
            precisions.append(precision_at_i)

    if len(precisions) == 0:
        return 0.0

    return np.mean(precisions)

# Calculate AP for each query
aps = []
for query_name, relevance in queries.items():
    print(f"\n{query_name}: {relevance}")
    print("-" * 60)

    precisions = []
    num_relevant = 0
    relevant_ranks = []

    for i, is_relevant in enumerate(relevance, start=1):
        if is_relevant == 1:
            num_relevant += 1
            precision_at_i = num_relevant / i
            precisions.append(precision_at_i)
            relevant_ranks.append(i)
            print(f"  Relevant doc at rank {i:2d}: Precision@{i} = {num_relevant}/{i} = {precision_at_i:.4f}")

    ap = np.mean(precisions) if precisions else 0
    aps.append(ap)

    print(f"\n  Average Precision (AP) = mean({[f'{p:.4f}' for p in precisions]})")
    print(f"                         = {ap:.4f}")

# Calculate MAP
print("\n" + "="*60)
print("FINAL: Mean Average Precision (MAP)")
print("="*60)

map_score = np.mean(aps)
print(f"\nMAP = mean(AP for all queries)")
print(f"    = mean([{', '.join([f'{ap:.4f}' for ap in aps])}])")
print(f"    = {map_score:.4f}")

print("\n✅ Key Insights:")
print("   - AP is high when relevant docs appear early in ranking")
print("   - Query 1 has best AP (relevant docs at ranks 1,2,4,7)")
print("   - Query 2 has worst AP (first relevant doc at rank 2)")
print("   - MAP averages AP across all queries")
print("   - Used for multi-document retrieval evaluation")


## Mean Reciprocal Rank (MRR)

- Measures effectiveness of retrieval system for queries with single correct answer
- Focuses on position of first relevant document in ranked results
- Calculated as reciprocal (1/rank) of first correct result
- Averages these values across all queries
- Higher MRR indicates better ranking of first relevant result

### Example
- If first relevant doc is at rank 1, score = 1/1 = 1.0
- If first relevant doc is at rank 3, score = 1/3 ≈ 0.33
- If first relevant doc is at rank 10, score = 1/10 = 0.1

### Formula

$$\text{MRR} = \frac{1}{N} \sum \frac{1}{\text{rank}_i}$$

where N is number of queries and $\text{rank}_i$ is position of first relevant doc for query i

## 📊 Mean Reciprocal Rank (MRR) - Detailed Calculation

MRR focuses on the **first relevant document** only.

**Use case**: When users typically want just ONE correct answer (e.g., "Who is the president of France?")

In [ ]:
# MRR (Mean Reciprocal Rank) Worked Example
import numpy as np

print("="*60)
print("Mean Reciprocal Rank (MRR) Example")
print("="*60)

# Scenario: 5 queries, each returns ranked results
# We only care about WHERE the FIRST relevant doc appears

queries_mrr = {
    "Who is the president of France?": 1,      # First relevant at rank 1
    "What is the capital of Japan?": 3,        # First relevant at rank 3
    "When did WWII end?": 1,                   # First relevant at rank 1
    "Who wrote Hamlet?": 2,                    # First relevant at rank 2
    "What is the speed of light?": 5,          # First relevant at rank 5
}

print("\nQueries and first relevant document rank:")
print("-" * 60)

reciprocal_ranks = []
for query, rank in queries_mrr.items():
    rr = 1.0 / rank
    reciprocal_ranks.append(rr)
    print(f"  '{query}'")
    print(f"    → First relevant at rank {rank} → RR = 1/{rank} = {rr:.4f}\n")

# Calculate MRR
print("="*60)
print("Calculating MRR")
print("="*60)

mrr = np.mean(reciprocal_ranks)
print(f"\nReciprocal Ranks: {[f'{rr:.4f}' for rr in reciprocal_ranks]}")
print(f"\nMRR = mean(Reciprocal Ranks)")
print(f"    = mean([{', '.join([f'{rr:.4f}' for rr in reciprocal_ranks])}])")
print(f"    = {mrr:.4f}")

# Compare different scenarios
print("\n" + "="*60)
print("MRR Interpretation Examples")
print("="*60)

scenarios = {
    "Perfect system (all rank 1)": [1, 1, 1, 1, 1],
    "Good system (ranks 1-2)": [1, 1, 2, 2, 1],
    "Mediocre system (ranks 1-5)": [1, 3, 2, 5, 4],
    "Poor system (ranks 5-10)": [5, 7, 6, 8, 10],
}

for scenario_name, ranks in scenarios.items():
    rrs = [1.0/r for r in ranks]
    mrr_scenario = np.mean(rrs)
    print(f"\n{scenario_name}:")
    print(f"  Ranks: {ranks}")
    print(f"  MRR = {mrr_scenario:.4f}")

print("\n✅ Key Insights:")
print("   - MRR = 1.0: Perfect (all first relevant docs at rank 1)")
print("   - MRR ≈ 0.5: Decent (avg first relevant at rank 2)")
print("   - MRR < 0.2: Poor (avg first relevant at rank 5+)")
print("   - Best for: Single-answer questions (QA, navigation)")
print("   - NOT for: Multi-document retrieval (use MAP instead)")


## Normalized Discounted Cumulative Gain (NDCG)

- Measures ranking quality for graded relevance scores
- Penalizes highly relevant documents appearing lower in results
- Accounts for multiple levels of relevance (e.g., 0-4 scale)
- Normalizes scores to allow comparison between queries
- Higher NDCG indicates better ranking of relevant documents
- Particularly useful when some results are more relevant than others

### Example
If relevance scores are [4,3,1,0] for top 4 results:
- DCG = 4 + 3/log₂(3) + 1/log₂(4)
- IDCG = scores sorted optimally [4,3,1,0]
- NDCG = DCG/IDCG

### Formula

$$\text{DCG} = \text{rel}_1 + \sum \frac{\text{rel}_i}{\log_2(i+1)}$$

$$\text{NDCG} = \frac{\text{DCG}}{\text{IDCG}}$$

where $\text{rel}_i$ is relevance of document at position i

## 📊 NDCG - Detailed Calculation

NDCG handles **graded relevance** (e.g., 0=irrelevant, 1=marginally relevant, 2=relevant, 3=highly relevant).

**Key idea**: Discount relevance scores by their position (log rank).

In [ ]:
# NDCG (Normalized Discounted Cumulative Gain) Worked Example
import numpy as np
import matplotlib.pyplot as plt

print("="*60)
print("Normalized Discounted Cumulative Gain (NDCG) Example")
print("="*60)

# Scenario: Product search query returns 5 items
# Relevance scores: 0=not relevant, 1=somewhat, 2=relevant, 3=highly relevant

print("\nRetrieved items (in rank order):")
retrieved_relevance = [3, 2, 1, 0, 2]  # Actual ranking
for i, rel in enumerate(retrieved_relevance, start=1):
    print(f"  Rank {i}: Relevance = {rel}")

# Step 1: Calculate DCG (Discounted Cumulative Gain)
print("\n" + "="*60)
print("STEP 1: Calculate DCG (Discounted Cumulative Gain)")
print("="*60)

def calculate_dcg(relevance_scores):
    """DCG = rel_1 + sum(rel_i / log2(i+1) for i >= 2)"""
    dcg = relevance_scores[0]  # First position not discounted
    for i, rel in enumerate(relevance_scores[1:], start=2):
        discount = np.log2(i + 1)
        dcg += rel / discount
    return dcg

dcg = 0
print(f"\nDCG formula: rel_1 + Σ(rel_i / log₂(i+1) for i ≥ 2)")
print(f"\nCalculation:")
print(f"  Position 1: rel = {retrieved_relevance[0]} (no discount) → {retrieved_relevance[0]:.4f}")
dcg += retrieved_relevance[0]

for i, rel in enumerate(retrieved_relevance[1:], start=2):
    discount = np.log2(i + 1)
    contribution = rel / discount
    dcg += contribution
    print(f"  Position {i}: rel = {rel}, discount = log₂({i+1}) = {discount:.4f} → {rel}/{discount:.4f} = {contribution:.4f}")

print(f"\n  DCG = {dcg:.4f}")

# Step 2: Calculate IDCG (Ideal DCG)
print("\n" + "="*60)
print("STEP 2: Calculate IDCG (Ideal DCG)")
print("="*60)

# Ideal ranking: sort by relevance (highest first)
ideal_relevance = sorted(retrieved_relevance, reverse=True)
print(f"\nIdeal ranking (sorted by relevance): {ideal_relevance}")

idcg = 0
print(f"\nCalculation:")
print(f"  Position 1: rel = {ideal_relevance[0]} (no discount) → {ideal_relevance[0]:.4f}")
idcg += ideal_relevance[0]

for i, rel in enumerate(ideal_relevance[1:], start=2):
    discount = np.log2(i + 1)
    contribution = rel / discount
    idcg += contribution
    print(f"  Position {i}: rel = {rel}, discount = log₂({i+1}) = {discount:.4f} → {rel}/{discount:.4f} = {contribution:.4f}")

print(f"\n  IDCG = {idcg:.4f}")

# Step 3: Calculate NDCG
print("\n" + "="*60)
print("STEP 3: Calculate NDCG = DCG / IDCG")
print("="*60)

ndcg = dcg / idcg if idcg > 0 else 0
print(f"\nNDCG = DCG / IDCG")
print(f"     = {dcg:.4f} / {idcg:.4f}")
print(f"     = {ndcg:.4f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

positions = np.arange(1, len(retrieved_relevance) + 1)

ax1.bar(positions, retrieved_relevance, color='steelblue', alpha=0.7)
ax1.set_xlabel('Rank Position')
ax1.set_ylabel('Relevance Score')
ax1.set_title(f'Actual Ranking\nDCG = {dcg:.3f}')
ax1.set_xticks(positions)
ax1.grid(axis='y', alpha=0.3)

ax2.bar(positions, ideal_relevance, color='green', alpha=0.7)
ax2.set_xlabel('Rank Position')
ax2.set_ylabel('Relevance Score')
ax2.set_title(f'Ideal Ranking\nIDCG = {idcg:.3f}')
ax2.set_xticks(positions)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle(f'NDCG = {ndcg:.3f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Key Insights:")
print(f"   - NDCG = {ndcg:.2f} means our ranking is {ndcg*100:.0f}% as good as ideal")
print("   - NDCG = 1.0: Perfect ranking (matches ideal)")
print("   - NDCG < 1.0: Suboptimal (relevant items ranked too low)")
print("   - Logarithmic discount: Position 2 matters much more than position 10")
print("   - Best for: Graded relevance (not binary relevant/irrelevant)")


## Evaluating the Retriever: When to Use Which Metric

**Requires labelled data!**

### MRR (Mean Reciprocal Rank)
- **Use when**: One single correct answer exists
- **Example**: "Who is the president of France?"
- **Focus**: Position of THE correct answer

### MAP (Mean Average Precision)
- **Use when**: Multiple equally relevant documents
- **Example**: All papers about "deep learning"
- **Focus**: Finding ALL relevant documents

### NDCG (Normalized Discounted Cumulative Gain)
- **Use when**: Documents have different relevance levels
- **Example**: Reviews rated 1-5 stars
- **Focus**: Ranking quality with graded relevance

## The Retrieval Stage in RAG

- Converts query to vector representation
- Searches indexed document/passage embeddings
- Uses dense or sparse retrieval methods
- Ranks results by relevance to query
- Selects top-k most relevant passages
- **Crucial for providing context to LLM**

## 🔧 Simple RAG Pipeline Implementation

Let's build a minimal end-to-end RAG system with:
1. Document indexing
2. Query → Retrieval
3. Retrieved context + Query → Generation (simulated)

In [ ]:
# Simple RAG Pipeline Implementation
from collections import defaultdict
import numpy as np

print("="*60)
print("Minimal RAG Pipeline")
print("="*60)

# Step 1: Document Collection
documents = [
    "The Eiffel Tower is located in Paris, France. It was built in 1889.",
    "The Statue of Liberty is in New York City. It was a gift from France in 1886.",
    "The Great Wall of China is over 13,000 miles long.",
    "The Colosseum in Rome was built in 80 AD.",
    "Mount Everest is the tallest mountain in the world at 29,032 feet.",
]

print("\n📚 Document Collection:")
for i, doc in enumerate(documents):
    print(f"  Doc {i}: {doc}")

# Step 2: Simple TF-IDF Indexing
print("\n" + "="*60)
print("STEP 1: Indexing Documents (TF-IDF)")
print("="*60)

from collections import Counter
import math

def tokenize(text):
    return text.lower().replace(',', '').replace('.', '').split()

# Build vocabulary and document frequency
vocab = set()
doc_tokens = []
for doc in documents:
    tokens = tokenize(doc)
    doc_tokens.append(tokens)
    vocab.update(tokens)

df = defaultdict(int)
for tokens in doc_tokens:
    for word in set(tokens):
        df[word] += 1

# Build TF-IDF vectors
def compute_tfidf_vector(tokens, vocab, df, N):
    tf = Counter(tokens)
    vector = {}
    for word in vocab:
        if word in tf:
            tfidf = tf[word] * math.log(N / df[word])
            vector[word] = tfidf
        else:
            vector[word] = 0
    return vector

N = len(documents)
tfidf_vectors = [compute_tfidf_vector(tokens, vocab, df, N) for tokens in doc_tokens]

print(f"✅ Indexed {len(documents)} documents with vocabulary size {len(vocab)}")

# Step 3: Query and Retrieve
print("\n" + "="*60)
print("STEP 2: Query → Retrieval")
print("="*60)

query = "Where is the Eiffel Tower?"
print(f"\nQuery: '{query}'")

query_tokens = tokenize(query)
query_vector = compute_tfidf_vector(query_tokens, vocab, df, N)

# Compute cosine similarity
def cosine_sim_dict(vec1, vec2):
    dot = sum(vec1.get(k, 0) * vec2.get(k, 0) for k in vocab)
    norm1 = math.sqrt(sum(v**2 for v in vec1.values()))
    norm2 = math.sqrt(sum(v**2 for v in vec2.values()))
    return dot / (norm1 * norm2) if norm1 > 0 and norm2 > 0 else 0

similarities = [cosine_sim_dict(query_vector, doc_vec) for doc_vec in tfidf_vectors]

# Rank and retrieve top-k
k = 2
top_k_indices = np.argsort(similarities)[::-1][:k]

print(f"\nTop-{k} Retrieved Documents:")
for rank, idx in enumerate(top_k_indices, start=1):
    print(f"  {rank}. Doc {idx} (similarity={similarities[idx]:.4f})")
    print(f"     {documents[idx]}\n")

# Step 4: Generation (Simulated)
print("="*60)
print("STEP 3: Retrieved Context + Query → Generation")
print("="*60)

retrieved_context = "\n".join([documents[idx] for idx in top_k_indices])

print(f"\nPrompt to LLM:")
print("-" * 60)
print(f"Context:\n{retrieved_context}")
print(f"\nQuestion: {query}")
print(f"\nAnswer based on the context above:")
print("-" * 60)

# Simulated generation (in real system, this would call an LLM)
simulated_answer = "The Eiffel Tower is located in Paris, France. It was built in 1889."

print(f"\n[Simulated LLM Response]")
print(f"→ {simulated_answer}")

print("\n✅ RAG Pipeline Complete!")
print("   1. Indexed documents with TF-IDF")
print("   2. Retrieved top-k most similar documents")
print("   3. Generated answer conditioned on retrieved context")


# Evaluating RAG for Question Answering Systems

## Evaluation Metrics

- **Exact Match**: Perfect answer match percentage
- **F1 Score**: Token overlap with gold answer
- **ROUGE**: N-gram overlap for longer answers
- **BERTScore**: Measure of Semantic Similarity
- **LLM-as-Judge**: LLM Measure of correctness
- **Human Evaluation**: Assesses subjective quality

## Exact Match

- Simplest evaluation: % of perfect matches
- Requires ground truth answer
- Binary: 1 for exact match, 0 otherwise
- Very strict (misses partial correctness)
- Best for factual, specific answers

**Example**: "New York City" matches but "NYC" does not

## F1 Score

- Measures token overlap with answer
- Requires ground truth answer
- Combines precision and recall of tokens
- More forgiving than exact match
- Good for short, structured answers

**Example**: "New York City" vs "NYC" gets partial credit

## ROUGE

**= Recall-Oriented Understudy for Gisting Evaluation**

- Measures n-gram overlap
- Requires ground truth answer
- Various types (ROUGE-1, ROUGE-L, etc.)
- Standard for summarization tasks
- Good for longer text responses

**Example**: Evaluating paragraph summaries

### Formula

$$\text{ROUGE-N} = \frac{\sum \text{Count}_{\text{match}}(\text{n-gram})}{\sum \text{Count}(\text{n-gram})}$$

Where:
- **n-gram**: Continuous sequence of n words
- **Count_match**: Overlapping n-grams
- **Count**: Total n-grams in reference
- n typically = 1, 2, or 4

## 📊 ROUGE Calculation Example

ROUGE measures n-gram overlap between generated and reference text.

In [ ]:
# ROUGE Calculation Example
from collections import Counter

print("="*60)
print("ROUGE (Recall-Oriented Understudy for Gisting Evaluation)")
print("="*60)

# Example: Summarization task
reference = "The dog chased the cat in the garden"
generated = "The dog chased a cat in the park"

print(f"\nReference: '{reference}'")
print(f"Generated: '{generated}'")

# Tokenize
ref_tokens = reference.lower().split()
gen_tokens = generated.lower().split()

print(f"\nReference tokens: {ref_tokens}")
print(f"Generated tokens: {gen_tokens}")

# ROUGE-1: Unigram overlap
print("\n" + "="*60)
print("ROUGE-1 (Unigram Overlap)")
print("="*60)

ref_unigrams = Counter(ref_tokens)
gen_unigrams = Counter(gen_tokens)

# Count overlapping unigrams
overlap = sum((gen_unigrams & ref_unigrams).values())
ref_count = len(ref_tokens)
gen_count = len(gen_tokens)

rouge1_recall = overlap / ref_count if ref_count > 0 else 0
rouge1_precision = overlap / gen_count if gen_count > 0 else 0
rouge1_f1 = 2 * (rouge1_precision * rouge1_recall) / (rouge1_precision + rouge1_recall) \
            if (rouge1_precision + rouge1_recall) > 0 else 0

print(f"\nOverlapping unigrams: {overlap}")
print(f"  Common words: {set(ref_tokens) & set(gen_tokens)}")
print(f"\nROUGE-1 Recall    = {overlap}/{ref_count} = {rouge1_recall:.4f}")
print(f"ROUGE-1 Precision = {overlap}/{gen_count} = {rouge1_precision:.4f}")
print(f"ROUGE-1 F1        = {rouge1_f1:.4f}")

# ROUGE-2: Bigram overlap
print("\n" + "="*60)
print("ROUGE-2 (Bigram Overlap)")
print("="*60)

def get_bigrams(tokens):
    return [tuple(tokens[i:i+2]) for i in range(len(tokens)-1)]

ref_bigrams = Counter(get_bigrams(ref_tokens))
gen_bigrams = Counter(get_bigrams(gen_tokens))

print(f"\nReference bigrams: {list(ref_bigrams.keys())}")
print(f"Generated bigrams: {list(gen_bigrams.keys())}")

overlap_bigrams = sum((gen_bigrams & ref_bigrams).values())
ref_bigram_count = sum(ref_bigrams.values())
gen_bigram_count = sum(gen_bigrams.values())

rouge2_recall = overlap_bigrams / ref_bigram_count if ref_bigram_count > 0 else 0
rouge2_precision = overlap_bigrams / gen_bigram_count if gen_bigram_count > 0 else 0
rouge2_f1 = 2 * (rouge2_precision * rouge2_recall) / (rouge2_precision + rouge2_recall) \
            if (rouge2_precision + rouge2_recall) > 0 else 0

print(f"\nOverlapping bigrams: {overlap_bigrams}")
print(f"  Common: {set(ref_bigrams.keys()) & set(gen_bigrams.keys())}")
print(f"\nROUGE-2 Recall    = {overlap_bigrams}/{ref_bigram_count} = {rouge2_recall:.4f}")
print(f"ROUGE-2 Precision = {overlap_bigrams}/{gen_bigram_count} = {rouge2_precision:.4f}")
print(f"ROUGE-2 F1        = {rouge2_f1:.4f}")

# ROUGE-L: Longest Common Subsequence
print("\n" + "="*60)
print("ROUGE-L (Longest Common Subsequence)")
print("="*60)

def lcs_length(s1, s2):
    """Calculate longest common subsequence length"""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    return dp[m][n]

lcs_len = lcs_length(ref_tokens, gen_tokens)

rougeL_recall = lcs_len / ref_count if ref_count > 0 else 0
rougeL_precision = lcs_len / gen_count if gen_count > 0 else 0
rougeL_f1 = 2 * (rougeL_precision * rougeL_recall) / (rougeL_precision + rougeL_recall) \
            if (rougeL_precision + rougeL_recall) > 0 else 0

print(f"\nLongest Common Subsequence length: {lcs_len}")
print(f"\nROUGE-L Recall    = {lcs_len}/{ref_count} = {rougeL_recall:.4f}")
print(f"ROUGE-L Precision = {lcs_len}/{gen_count} = {rougeL_precision:.4f}")
print(f"ROUGE-L F1        = {rougeL_f1:.4f}")

print("\n✅ ROUGE Summary:")
print(f"   ROUGE-1 F1: {rouge1_f1:.3f} (word overlap)")
print(f"   ROUGE-2 F1: {rouge2_f1:.3f} (phrase overlap)")
print(f"   ROUGE-L F1: {rougeL_f1:.3f} (sequential overlap)")


## BERTScore

- Semantic similarity using BERT embeddings
- Requires ground truth answer
- Captures meaning beyond exact words
- More nuanced than lexical overlap
- Good for paraphrased answers

**Example**: "The dog is quick" ≈ "The canine is fast"

### How it works
1. **Contextual Embedding**: Generate BERT embeddings for each token
2. **Pairwise Cosine Similarity**: Compute similarity matrix
3. **Maximum Similarity**: For each token, find best match
4. **Aggregate Scores**:

$$R_{\text{BERT}} = \frac{1}{|\hat{x}|} \sum_{\hat{x}_j \in \hat{x}} \max_{x_i \in x} \mathbf{x}_i^\top \hat{\mathbf{x}}_j$$

$$P_{\text{BERT}} = \frac{1}{|x|} \sum_{x_i \in x} \max_{\hat{x}_j \in \hat{x}} \mathbf{x}_i^\top \hat{\mathbf{x}}_j$$

$$F_{\text{BERT}} = 2 \frac{P_{\text{BERT}} \cdot R_{\text{BERT}}}{P_{\text{BERT}} + R_{\text{BERT}}}$$

## LLM-as-Judge

- Uses LLM to evaluate responses
- Requires ground truth answer (usually)
- Can assess reasoning & correctness
- More flexible than automatic metrics
- **Potentially inconsistent**

### Limitation
> "Limited evidence that 11 state-of-the-art LLMs are ready to replace expert or non-expert human judges" (Bavaresco 2024)

### LLM Evaluation Metric Workflow
```
LLM Test Case
(Input, LLM Output, 
 Retrieval Context, etc.)
         ↓
    LLM Judge Scorer
         ↓
  Passes threshold?
         ↓
Score / Reason / Metric
```

## Human Evaluation

- **Gold standard** for subjective quality
- May need ground truth for reference
- Most comprehensive assessment
- Resource-intensive & slower
- Can capture nuance & creativity

**Example**: Rating helpfulness 1-5

## Faithfulness in RAG Systems

**Faithfulness** measures how true RAG systems are to their retrieved context

- It does **not** verify that the retrieved context is correct
- The rate at which claims in the output are entailed by claims in the retrieved chunks

### Definition

$$\text{Faithfulness} = \frac{\text{claims\_supported\_by\_context}}{\text{total\_claims}}$$

### Example

**Context:** "The Eiffel Tower was built in 1889"

**Response:** "The Eiffel Tower was built in 1889 and is in London"

**Faithfulness = 1/2** (First claim true to context, second not)

## 📊 Faithfulness Calculator

Faithfulness measures how well the generated answer is grounded in the retrieved context.

In [ ]:
# Faithfulness Calculator
print("="*60)
print("Faithfulness in RAG Systems")
print("="*60)

# Example scenario
context = """
The Eiffel Tower is located in Paris, France. It was built in 1889 for the
World's Fair. The tower is 330 meters tall and was designed by Gustave Eiffel.
"""

# Two generated responses
response_faithful = """
The Eiffel Tower is in Paris, France. It was built in 1889 and stands 330 meters tall.
"""

response_unfaithful = """
The Eiffel Tower is in Paris, France. It was built in 1889 and is the tallest structure
in the world. It was designed by Leonardo da Vinci.
"""

print("\nRetrieved Context:")
print("-" * 60)
print(context.strip())

# Response 1: Faithful
print("\n" + "="*60)
print("Response 1 (Faithful)")
print("="*60)
print(response_faithful.strip())

print("\nClaims extracted:")
claims_1 = [
    ("Eiffel Tower is in Paris, France", True, "✓ Supported by context"),
    ("Built in 1889", True, "✓ Supported by context"),
    ("330 meters tall", True, "✓ Supported by context"),
]

for claim, supported, reason in claims_1:
    status = "✓" if supported else "✗"
    print(f"  {status} '{claim}' - {reason}")

faithfulness_1 = sum(1 for _, s, _ in claims_1 if s) / len(claims_1)
print(f"\nFaithfulness = {sum(1 for _, s, _ in claims_1 if s)}/{len(claims_1)} = {faithfulness_1:.2f} (100%)")

# Response 2: Unfaithful
print("\n" + "="*60)
print("Response 2 (Unfaithful)")
print("="*60)
print(response_unfaithful.strip())

print("\nClaims extracted:")
claims_2 = [
    ("Eiffel Tower is in Paris, France", True, "✓ Supported by context"),
    ("Built in 1889", True, "✓ Supported by context"),
    ("Tallest structure in the world", False, "✗ NOT in context (hallucination)"),
    ("Designed by Leonardo da Vinci", False, "✗ Contradicts context (context says Gustave Eiffel)"),
]

for claim, supported, reason in claims_2:
    status = "✓" if supported else "✗"
    print(f"  {status} '{claim}' - {reason}")

faithfulness_2 = sum(1 for _, s, _ in claims_2 if s) / len(claims_2)
print(f"\nFaithfulness = {sum(1 for _, s, _ in claims_2 if s)}/{len(claims_2)} = {faithfulness_2:.2f} (50%)")

# Summary
print("\n" + "="*60)
print("Key Distinctions")
print("="*60)

print("""
✅ Faithful Response:
   - All claims can be verified from retrieved context
   - May miss information, but doesn't add false info
   - High faithfulness = trustworthy

❌ Unfaithful Response:
   - Contains claims NOT in context (hallucination)
   - May contradict context (fabrication)
   - Low faithfulness = untrustworthy

⚠️  Important Notes:
   - Faithfulness ≠ Correctness
   - Context itself might be wrong!
   - Faithfulness only measures: "Is answer grounded in retrieved docs?"
   - Separate concern: "Are the retrieved docs correct?"
""")

print("✅ Formula:")
print("   Faithfulness = (# supported claims) / (# total claims)")


# Advanced RAG Topics

## End-to-End RAG Training

Train the retriever and generator to improve accuracy:

### Generator
Maximize generation likelihood given single retrieved document

### Retriever
Maximize overall likelihood by optimizing mixture weights over documents

### Training Process
```
Question → Query Encoder → q(x)
                              ↓
Documents → Document Index → d(z)
                              ↓
                            MIPS  → Top-k docs
                              ↓
              Generator pθ → Answer
```

**Example Question:**
- "Barack Obama was born in Hawaii."
- "The Divine Comedy."
- "Jeopardy Question Generation: Answer Query"

## End-to-End Training Equations

### Generation is a mixture model
Pick a document, generate from the document:

$$p_{\text{RAG-Token}}(y|x) \approx \prod_i \sum_{z \in \text{top-}k(p(\cdot|x))} p_\eta(z|x) p_\theta(y_i|x, z, y_{1:i-1})$$

### Probability of the context is based on embeddings

$$p_\eta(z|x) \propto \exp(\mathbf{d}(z)^\top \mathbf{q}(x)), \quad \mathbf{d}(z) = \text{enc}_d(z), \quad \mathbf{q}(x) = \text{enc}_q(x)$$

- Adjusts retriever to give higher similarities to helpful docs
- Can't update document embeddings

**Question:** Is this a bi-encoder or a cross-encoder?

**Answer:** This is a **bi-encoder** - separate encoders for query and document!

## When to Retrieve?

### Timing Options

- **Default RAG assumption**: "At the start" (Lewis et al. 2020)

- **Several times during generation**, as necessary:
  - Let the model invoke the search (Schick et al. 2023)
  - Search when the model is uncertain (Jiang et al. 2023)

- **Every turn?**

- **Multiple retrievals per "search"**

### Query Transformation
```
query → LLM → subquery 1 → Vector index → Top k results
           → subquery 2 or more → Vector index → Top k results
           → answer
```

# Challenges in RAG Implementation

## Key Challenges

- Balancing retrieval speed and accuracy
- Handling multi-hop or complex queries
- Ensuring consistency across multiple passages
- Managing large-scale document collections efficiently
- Adapting to domain-specific vocabulary and context
- Optimizing prompt design for varied queries

## Privacy Concerns in RAG

- **Proprietary data may leak to LLM providers**
- Sensitive information in retrieval results
- Data retention policies for queries/responses
- User consent for data usage unclear
- Re-identification risks from aggregated data
- Compliance challenges with data protection laws

### Real-World Examples

**Samsung Bans ChatGPT Among Employees After Sensitive Code Leak**

**Microsoft Copilot could have serious vulnerabilities after researchers reveal data leak issues in RAG systems**

## Addressing Hallucination in LLMs

- **RAG**: Ground responses in retrieved information
- **Calibration**: Improve model uncertainty estimates
- **Multi-source verification**: Cross-check multiple sources
- **Factuality checks**: Post-processing for accuracy
- **Prompt engineering**: Encourage source attribution
- **Fine-tuning**: Optimize for truthful responses

## RAG isn't perfect!

### Key Limitations

- **LLMs struggle to attend to multiple documents**
  - Performance degrades with more retrieved documents
  
- **Citation correctness is low**
  - Models often cite wrong sources
  
- **Order of retrieved context matters**
  - LLMs attend more to most recent context (recency bias)

### Performance vs Number of Retrieved Docs

Graph shows that:
- claude-1.3-100k (orange) maintains ~85-90% performance
- Most other models plateau at 50-60% accuracy
- Performance generally increases from 5 to 20 docs, then plateaus

# Future Directions in RAG Research

- **Generation of structured knowledge bases**: GraphRAG
- **Use of multiple indexes**
- **Reranking of responses**
- **Divergence-based measures of faithfulness**
- **Dynamic updating of retrieval databases**
- **Personalization in retrieval and generation**
- **Cross-modal RAG for multimodal tasks**

## Summary and Key Takeaways

### Question Answering in NLP
- Systems that answer questions posed in natural language
- Many types: factoid, conversational, multi-hop, visual, etc.
- Reading comprehension as a testbed for language understanding

### LLMs as Knowledge Bases
- Zero-shot learners with scaling
- Parametric knowledge from training data
- Limitations: biased, hallucinate, frozen, privacy concerns

### Retrieval Augmented Generation (RAG)
- Combines Information Retrieval + Natural Language Generation
- Components: Document Collection, Retriever, Prompt Engineering, LLM, Evaluation
- Addresses LLM limitations by grounding in external knowledge

### Information Retrieval Methods
- **Sparse**: TF-IDF, BM25
- **Dense**: Bi-encoders (efficient), Cross-encoders (accurate)
- Approximate nearest neighbor search for scalability

### Evaluation Metrics
- **Retrieval**: Precision, Recall, MAP, MRR, NDCG
- **Generation**: Exact Match, F1, ROUGE, BERTScore, LLM-as-Judge
- **RAG-specific**: Faithfulness

### Challenges and Future Directions
- Multi-document attention
- Privacy and hallucination
- When to retrieve
- GraphRAG, multiple indexes, personalization

## References and Further Reading

### Key Papers

- Lewis et al. (2020) - "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"
- Rajpurkar et al. (2016) - "SQuAD: 100,000+ Questions for Machine Comprehension of Text"
- Gao et al. (2024) - "Retrieval-Augmented Generation for Large Language Models: A Survey"
- Yu (2024) - "RAG Evaluation Metrics"
- Farahani (2024) - "When LLMs Use Retrieved Context"
- Bavaresco (2024) - "LLM-as-Judge Evaluation"

### Datasets

- SQuAD - Stanford Question Answering Dataset
- CoQA - Conversational Question Answering
- MMLU - Multi-task Multi-domain Language Understanding
- HotpotQA - Multi-hop Question Answering
- ELI5 - Long Form Question Answering

### Tools and Libraries

- LlamaIndex - RAG framework
- FAISS - Efficient similarity search
- Sentence Transformers - Dense retrieval models

## Thank You!

### Questions?

Please email with any feedback or questions.

---

**Next Lecture**: Knowledge Graphs and Semantic Web

**Recommended Reading**: 
- Lewis et al. (2020) - RAG paper
- Gao et al. (2024) - RAG survey